In [0]:
display(dbutils.fs.ls('/databricks-datasets/cs110x/ml-20m/data-001/'))

In [0]:
display(dbutils.fs.ls('/databricks-datasets/cs110x/ml-20m/data-001/'))

In [0]:
%fs head /databricks-datasets/cs110x/ml-20m/data-001/movies.csv

In [0]:
from pyspark.sql.types import *

movies_schema = StructType([
  StructField('movieId', IntegerType()),
  StructField('title', StringType()),
  StructField('genres', StringType())
])
ratings_schema = StructType([
  StructField('userId', IntegerType()),
  StructField('movieId', IntegerType()),
  StructField('rating', FloatType()), 
])



In [0]:
file_location = "/databricks-datasets/cs110x/ml-20m/data-001/movies.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "false"


# The applied options are for CSV files. For other file types, these will be ignored.
df_movies = spark.read.format(file_type) \
  .option("inferSchema", "false") \
  .option("header", "true") \
  .schema(movies_schema) \
  .load(file_location)

display(df_movies)

In [0]:
file_location = "/databricks-datasets/cs110x/ml-20m/data-001/ratings.csv"
file_type = "csv"


infer_schema = "true"
first_row_is_header = "false"


df_ratings = spark.read.format(file_type) \
  .option("inferSchema", "false") \
  .option("header", "true") \
  .schema(ratings_schema) \
  .load(file_location)

df_rating_train, df_rating_test = df_ratings.randomSplit([0.8, 0.2], seed=42)



In [0]:
from pyspark.sql.functions import count, col, lit, desc
from pyspark.sql.types import *
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
ranks = [10, 15, 20]
regParams = [0.01, 0.05, 0.1]
iterations = [10, 15, 20]

results = []
best_rmse = float("inf")
best_params = {}

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

for r in ranks:
    for reg in regParams:
        for iter_val in iterations:
            als = ALS(rank=r, 
                      maxIter=iter_val, 
                      regParam=reg, 
                      userCol="userId", 
                      itemCol="movieId", 
                      ratingCol="rating",
                      coldStartStrategy="drop") 
            
            model = als.fit(df_rating_train)
            
            predictions = model.transform(df_rating_test)
            rmse = evaluator.evaluate(predictions)
            
            results.append((r, reg, iter_val, rmse))
            
            print(f"Rank: {r} | Reg: {reg:<4} | Iters: {iter_val} | RMSE: {rmse:.4f}")
            
            if rmse < best_rmse:
                best_rmse = rmse
                best_params = {"rank": r, "regParam": reg, "maxIter": iter_val}

print("-" * 50)
print(f"Best RMSE: {best_rmse:.4f}")
print(f"Best Params: {best_params}")

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

df_results = pd.DataFrame(results, columns=["Rank", "RegParam", "MaxIter", "RMSE"])

sns.set_style("whitegrid")

g = sns.relplot(
    data=df_results, 
    x="Rank", 
    y="RMSE", 
    hue="RegParam",
    col="MaxIter",
    kind="line", 
    marker="o", 
    palette="flare",
    height=5, 
    aspect=1.2,
    linewidth=2.5
)

g.fig.suptitle("RMSE vs Rank", y=1.05, fontsize=16)
g.set_axis_labels("Rank", "RMSE")

g._legend.set_title("Regularization")

plt.show()

In [0]:
heatmap_data = df_results.pivot_table(index='Rank', columns='RegParam', values='RMSE', aggfunc='mean')

plt.figure(figsize=(6, 5))

sns.heatmap(heatmap_data, annot=True, fmt=".4f", cmap="flare", cbar_kws={'label': 'Average RMSE'})

plt.title("Finding the Lowest RMSE")
plt.xlabel("Regularization Parameter")
plt.ylabel("Rank")
plt.show()

In [0]:
my_user_id = 0
my_rated_movies = [
    (my_user_id, 364, 5), 
    (my_user_id, 520, 4), 
    (my_user_id, 521, 5), 
    (my_user_id, 875, 4), 
    (my_user_id, 1036, 5), 
    (my_user_id, 734, 3), 
    (my_user_id, 1054, 4), 
    (my_user_id, 904, 4),
    (my_user_id, 760, 5),  
    (my_user_id, 1243, 5), 
]
df_custom_ratings = spark.createDataFrame(my_rated_movies, ["userId", "movieId", "ratings"])
display(df_custom_ratings)

In [0]:


df_combined_ratings = df_ratings.select("userId", "movieId", "rating").union(df_custom_ratings)

best_rank = best_params.get("rank", 40)
best_reg = best_params.get("regParam", 0.05)
best_iter = best_params.get("maxIter", 20)

als_final = ALS(rank=best_rank, 
                maxIter=best_iter, 
                regParam=best_reg, 
                userCol="userId", 
                itemCol="movieId", 
                ratingCol="rating",
                coldStartStrategy="drop")

final_model = als_final.fit(df_combined_ratings)

df_rated_movie_ids = df_custom_ratings.select("movieId")
df_movies_unrated = df_movies.join(df_rated_movie_ids, on="movieId", how="left_anti")

df_movie_counts = df_rating_train.groupBy("movieId").agg(count("rating").alias("rating_count"))

df_candidates = df_movies_unrated.join(df_movie_counts, "movieId")

# Added a quick filter to remove obscure movies with very few ratings. I found the recommendations to be better with this threshold. Optional tweak!
df_candidates_filtered = df_candidates.filter(col("rating_count") > 100) 

df_prediction_input = df_candidates_filtered.withColumn("userId", lit(my_user_id))

predictions = final_model.transform(df_prediction_input)

top_20 = predictions.filter(col("prediction") != float('nan')) \
                    .orderBy(desc("predictions")) \
                    .select("title", "genres", "prediction", "rating_count") \
                    .limit(20)

display(top_20)